# Week 7 build · Probe a small language model and log the run

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-07.ipynb)

**~3 hours · CPU is enough (a T4 only for the memory measurement in Part D) · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two passes.** First run with `SMOKE = True` (next cell): a 14M-parameter model, 200 sentences, under five minutes on CPU, so you see every cell work. Then set `SMOKE = False` and run again for real: SmolLM2-135M and 800 sentences (a 270 MB download, then a few minutes even on CPU).

| Part | Topic | Time |
|---|---|---|
| A | Load the checkpoint; read the config; count the parameters | 20 min |
| B | Shapes in, shapes out: padding, hidden states, generation | 30 min |
| C | The linear probe, layer by layer, then with shuffled labels | 50 min |
| D | Memory arithmetic, measured on a GPU if you have one | 30 min |
| E | Reproducibility: seeds, recomputed features, the run log | 30 min |
| F | Stretch: last-token pooling | 20 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Load the checkpoint (20 min)

**Words used in this part.** *checkpoint files*: `config.json` (architecture as numbers), the weights file, the tokenizer files. *config*: those numbers, loaded as `model.config`. *parameter count*: total learned weights; week 5's rule is about `12·d²` per block plus the embedding table `V·d`.

In [ ]:
#@title Config: SMOKE = True is a five-minute CPU rehearsal; False is the real run
SMOKE = True  #@param {type:"boolean"}
import json, time, os, csv, subprocess, urllib.request, torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = dict(model="EleutherAI/pythia-14m" if SMOKE else "HuggingFaceTB/SmolLM2-135M",
           n_examples=200 if SMOKE else 800, seed=0, steps=300, lr=1e-2, split=0.8, pool="mean")
torch.manual_seed(cfg["seed"])
print("device:", device, "| torch", torch.__version__, "| transformers", transformers.__version__)
print("cfg:", cfg)

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg["model"])
model = AutoModelForCausalLM.from_pretrained(cfg["model"]).float().to(device).eval()   # .float(): 32-bit on every runtime
c = model.config
print(type(model).__name__, "| layers", c.num_hidden_layers, "| width", c.hidden_size, "| vocab", c.vocab_size)
print("MLP width", c.intermediate_size, "| heads", c.num_attention_heads, "| tied output layer:", c.tie_word_embeddings)

In [ ]:
#@title 🔮 Predict A1: Week 5's rule (12·d²·L + V·d) vs the real parameter count printed next: which is larger, and by roughly what factor? (use the config above)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
N = sum(p.numel() for p in model.parameters())
d, L, V = c.hidden_size, c.num_hidden_layers, c.vocab_size
rule = 12 * d * d * L + V * d
print(f"actual {N/1e6:.2f} M   week-5 rule {rule/1e6:.2f} M   actual / rule = {N/rule:.2f}")
for name, p in list(model.named_parameters())[:5]:
    print(f"  {name:48s} {tuple(p.shape)}")
print(f"weights on disk: {N*2/1e6:.0f} MB in 16-bit, {N*4/1e6:.0f} MB in 32-bit")

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

Compare the printed `MB in 16-bit` with the size of `model.safetensors` on the model's Hub page: that file-size arithmetic tells you the stored dtype before you download anything.

---
## Part B · Shapes in, shapes out (30 min)

**Words used in this part.** *input_ids*: the tokenizer's integer ids, batch × tokens. *attention_mask*: 1 on real tokens, 0 on padding. *hidden state*: one vector per token per layer, batch × tokens × width. *logits*: one score per vocabulary entry per position.

In [ ]:
#@title 🔮 Predict B1: Two sentences of different length, padding=True, through this model. Shapes of input_ids and of one hidden state, and how many hidden states come back?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
texts = ["Gravity is", "The haptic glove buzzed twice and then stopped ."]
try:
    tok(texts, return_tensors="pt", padding=True)
except ValueError as e:
    print("ERROR:", str(e)[:80], "...")
tok.pad_token = tok.eos_token                     # the standard one-line fix for decoder models
enc = tok(texts, return_tensors="pt", padding=True).to(device)
print({k: tuple(v.shape) for k, v in enc.items()}, "| padding side:", tok.padding_side)
print("row 0 tokens:", tok.convert_ids_to_tokens(enc["input_ids"][0]))

In [ ]:
with torch.no_grad():
    out = model(**enc, output_hidden_states=True)
print("hidden_states:", len(out.hidden_states), "x", tuple(out.hidden_states[0].shape), "= (layers + 1) x (B, T, d)")
print("logits:", tuple(out.logits.shape), "= (B, T, V)")
print(f"this batch's logits alone: {out.logits.numel() * 4 / 1e6:.1f} MB in 32-bit")
last_real = enc["attention_mask"].sum(1) - 1        # index of the last real token per row
print("last real token per row:", last_real.tolist(), "<- position -1 would be padding for row 0")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict B2: Sample 20 tokens after 'The haptic glove' twice with the same seed at temperature 0.8. Same text both times? Will it be English?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
prompt = tok(["The haptic glove"], return_tensors="pt").to(device)
for run in range(2):
    torch.manual_seed(cfg["seed"])
    g = model.generate(**prompt, max_new_tokens=20, do_sample=True, temperature=0.8, pad_token_id=tok.eos_token_id)
    print(f"run {run}:", repr(tok.decode(g[0][prompt["input_ids"].shape[1]:], clean_up_tokenization_spaces=False)))

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

Same seed, same tokens: the seed pins every random draw. With `SMOKE = True` the 14M model's English is rough; SmolLM2-135M in the real pass is noticeably better. That difference is the scaling law from week 6, felt.

---
## Part C · The linear probe (50 min)

**Words used in this part.** *linear probe*: one linear layer trained on a frozen model's hidden states to predict a label. *frozen weights*: the model receives no gradient. *pooling*: turning one vector per token into one vector per sentence, here the mean over real tokens. *held-back*: the 20 % the probe never trains on.

Data: sentences from the SST-2 sentiment set (label 1 = positive), fetched as JSON from the Hub's dataset viewer, no extra library.

In [ ]:
def fetch_sst2(n, split="validation"):
    base = "https://datasets-server.huggingface.co/rows?dataset=stanfordnlp/sst2&config=default&split=" + split
    rows = []
    for off in range(0, n, 100):
        url = f"{base}&offset={off}&length={min(100, n - off)}"
        rows += json.load(urllib.request.urlopen(url, timeout=60))["rows"]
    return [r["row"]["sentence"].strip() for r in rows], [r["row"]["label"] for r in rows]

texts, labels = fetch_sst2(cfg["n_examples"])
labels = torch.tensor(labels)
print(len(texts), "sentences |", f"{labels.float().mean():.2f} positive")
for t, l in list(zip(texts, labels))[:3]:
    print(l.item(), "|", t)

In [ ]:
@torch.no_grad()
def features(texts, bs=16):
    feats = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        hs = model(**enc, output_hidden_states=True).hidden_states          # (layers + 1) x (B, T, d)
        m = enc["attention_mask"][..., None].float()                          # (B, T, 1): 1 on real tokens
        feats.append(torch.stack([(h * m).sum(1) / m.sum(1) for h in hs]).cpu())   # mean over real tokens
    return torch.cat(feats, 1)                                                # (layers + 1, N, d)

t0 = time.time()
X = features(texts)
print("features:", tuple(X.shape), "= (layers + 1, examples, width)", f"in {time.time() - t0:.1f} s")

In [ ]:
def probe(feat, labels, seed, steps=300, lr=1e-2, split=0.8):
    """One linear layer on frozen features. Returns (train accuracy, held-back accuracy)."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(labels), generator=g)
    n_tr = int(split * len(labels))
    tr, te = idx[:n_tr], idx[n_tr:]
    mu, sd = feat[tr].mean(0), feat[tr].std(0) + 1e-6           # standardise with train statistics only
    Z = (feat - mu) / sd
    torch.manual_seed(seed)
    lin = torch.nn.Linear(feat.shape[1], 2)                      # the only trainable part
    opt = torch.optim.Adam(lin.parameters(), lr=lr, weight_decay=1e-2)
    for _ in range(steps):
        loss = torch.nn.functional.cross_entropy(lin(Z[tr]), labels[tr])
        opt.zero_grad(); loss.backward(); opt.step()
    acc = lambda ix: (lin(Z[ix]).argmax(1) == labels[ix]).float().mean().item()
    return acc(tr), acc(te)

In [ ]:
#@title 🔮 Predict C1: A probe per layer (80/20 split). Which layer scores best on the held-back 20%, roughly what accuracy, and how big is the train-vs-held-back gap?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
import matplotlib.pyplot as plt
results = []
for layer in range(X.shape[0]):
    tr_acc, te_acc = probe(X[layer], labels, cfg["seed"], cfg["steps"], cfg["lr"], cfg["split"])
    results.append((layer, tr_acc, te_acc))
    print(f"layer {layer:2d}   train {tr_acc:.2f}   held-back {te_acc:.2f}")
best = max(results, key=lambda r: r[2])
print(f"best layer {best[0]} held-back {best[2]:.2f} | chance 0.50")
plt.plot([r[0] for r in results], [r[1] for r in results], "o--", label="train")
plt.plot([r[0] for r in results], [r[2] for r in results], "o-", label="held-back")
plt.axhline(0.5, color="gray", lw=1); plt.xlabel("layer"); plt.ylabel("probe accuracy"); plt.legend(); plt.show()

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Two readings. The *held-back* curve is the one a paper reports. The *train* curve near 1.0 is not a result: with `d` features and a few hundred examples a linear layer can memorise, which is why the split exists. Now break the link between text and label and see what the probe does with pure noise:

In [ ]:
#@title 🔮 Predict C2: Shuffle the labels (text and label now unrelated), re-fit on the best layer. Train accuracy and held-back accuracy?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
g = torch.Generator().manual_seed(cfg["seed"] + 1)
shuffled = labels[torch.randperm(len(labels), generator=g)]
layer = best[0]
tr_s, te_s = probe(X[layer], shuffled, cfg["seed"], cfg["steps"], cfg["lr"], cfg["split"])
print(f"layer {layer} with real labels:      train {best[1]:.2f}   held-back {best[2]:.2f}")
print(f"layer {layer} with shuffled labels:  train {tr_s:.2f}   held-back {te_s:.2f}   <- chance is 0.50")

In [ ]:
#@title ✍️ C2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C2", what_happened, surprise)

If the shuffled held-back number sits near 0.50 while train stays high, the probe *can* memorise noise and the split *does* catch it. That pair of numbers is the check to demand of any probing claim.

---
## Part D · Memory arithmetic (30 min)

**Words used in this part.** *memory footprint*: bytes on the GPU. Inference in 16-bit: 2 per parameter. Full fine-tuning with Adam in 32-bit: 4 (weight) + 4 (gradient) + 8 (two Adam averages) = 16 per parameter, plus activations. *fp16 / bf16*: 16-bit number formats; the T4 has no bf16 hardware.

In [ ]:
#@title 🔮 Predict D1: For this model (N above), how many MB does full fine-tuning with Adam need in 32-bit, before activations? And does SmolLM2-1.7B fit a 15 GB T4?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
def footprint_gb(n_params, bytes_per_weight, training):
    per = bytes_per_weight + (4 + 8 if training else 0)       # + gradient (32-bit) + two Adam averages
    return n_params * per / 1e9

print(f"{'model':16s} {'infer 16-bit':>13s} {'fine-tune 32-bit':>17s}   fits a 15 GB T4?")
for name, n in [("this model", N), ("SmolLM2-360M", 362e6), ("SmolLM2-1.7B", 1.71e9), ("Llama-3 8B", 8.03e9)]:
    ft = footprint_gb(n, 4, True)
    print(f"{name:16s} {footprint_gb(n, 2, False):10.2f} GB {ft:14.2f} GB   {'yes' if ft < 15 else 'no'}")
print(f"transformers says this model is {model.get_memory_footprint() / 1e6:.0f} MB (weights + buffers, current dtype)")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

**GPU only.** One real fine-tuning step, then compare the measured peak with the prediction. On Colab: Runtime → Change runtime type → T4. On CPU the cell just says so.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    enc = tok(texts[:8], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
    out = model(**enc, labels=enc["input_ids"])          # labels= : next-token loss on your own text
    out.loss.backward(); opt.step(); opt.zero_grad()      # one full fine-tuning step
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"peak {peak:.2f} GB vs predicted {footprint_gb(N, 4, True):.2f} GB + activations | loss {out.loss.item():.3f}")
    print("bf16 supported here:", torch.cuda.is_bf16_supported())
    del opt, out; model.eval(); torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(cfg["model"]).float().to(device).eval()   # undo the step
else:
    print("not executed: no GPU in this runtime (Runtime -> Change runtime type -> T4)")

---
## Part E · Reproducibility (30 min)

**Words used in this part.** *random seed*: the integer every random draw starts from. *git hash*: the id of the code version, from `git rev-parse`. *run log*: one CSV row per run with time, code version, config, seed, versions and the result.

In [ ]:
#@title 🔮 Predict E1: Run the probe on the best layer with seed 0 twice, then seeds 1 and 2. Do the two seed-0 runs match to 4 decimals? How wide is the spread across seeds 0-2?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
layer = best[0]
runs = [(s, probe(X[layer], labels, s, cfg["steps"], cfg["lr"], cfg["split"])[1]) for s in [0, 0, 1, 2]]
for s, a in runs:
    print(f"seed {s}: held-back {a:.4f}")
accs = [a for _, a in runs[1:]]
print("same seed identical:", runs[0][1] == runs[1][1], f"| spread over seeds 0-2: {max(accs) - min(accs):.3f}")
print("note: the seed also chooses the 80/20 split, so the spread includes which sentences were held back")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E2: Recompute the features for the first 16 sentences on the same machine. Largest absolute difference from the stored features: exactly 0, or small but nonzero?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
X2 = features(texts[:16])
print("largest difference:", (X2 - X[:, :16]).abs().max().item(), "| device:", device)

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

In [ ]:
def git_hash():
    try:
        h = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip() != ""
        return h + ("-dirty" if dirty else "")
    except Exception:
        return "no-git"                                     # Colab has no repository: log that fact, not a fake hash

def log_run(seed, held_back_acc, path="week07_runs.csv"):
    row = dict(time=time.strftime("%Y-%m-%dT%H:%M:%S"), git=git_hash(), **{**cfg, "seed": seed}, layer=int(layer),
               device=device, torch=torch.__version__, transformers=transformers.__version__, held_back_acc=round(held_back_acc, 4))
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row))
        if new: w.writeheader()
        w.writerow(row)

for s, a in runs:
    log_run(s, a)
print(open("week07_runs.csv").read())

Four rows, one per run, and the two seed-0 rows agree. If `git` says `no-git` you are on Colab: paste the hash of the LearnML commit you are working from into the row by hand, or run the notebook from a clone. That column is what lets someone else check out the same code.

---
## Part F · Stretch: last-token pooling (20 min)

**Words used in this part.** *last-token pooling*: use the hidden state of each sentence's final real token instead of the mean. In a decoder model that position has seen the whole sentence.

In [ ]:
#@title 🔮 Predict F1: Last real token instead of the mean, on the best layer. Better or worse held-back accuracy than the mean, and by how much?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
@torch.no_grad()
def features_last(texts, layer, bs=16):
    feats = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        h = model(**enc, output_hidden_states=True).hidden_states[layer]
        last = enc["attention_mask"].sum(1) - 1                  # index of the last real token per row
        feats.append(h[torch.arange(len(last)), last].cpu())
    return torch.cat(feats)

X_last = features_last(texts, layer)
te_last = probe(X_last, labels, cfg["seed"], cfg["steps"], cfg["lr"], cfg["split"])[1]
print(f"layer {layer}: mean pooling {runs[0][1]:.3f} | last token {te_last:.3f}")

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-tooling`, and mark the biggest surprise.
Then on the laptop:

```
python track.py done week-07/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 8
```

In [ ]:
reveal()